In [2]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
import re
from statistics import mean
from anthropic.types import TextBlock, ThinkingBlock

client = Anthropic()
model = "claude-haiku-5-5" #"claude-sonnet-5-5"

In [11]:
def remove_json_markdown(response: str) -> str:
    ''' Removes the Markdown formatting from a JSON string '''

    m = re.search(r'^```json\n(.+)```$', response, re.DOTALL) #, re.MULTILINE | re.DOTALL)
    m1 = m.group(1)
    return m1

def response_content_text(response_content: list) -> str:

    if isinstance(response_content[0], TextBlock):
        print("The response contains a TextBlock.")
        response_text = response_content[0].text

    elif isinstance(response_content[0], ThinkingBlock):
        print("The response contains a ThinkingBlock.")
        removed_json_markdown = remove_json_markdown(response_content[1].text)
        response_text = removed_json_markdown

    print(f'RESPONSE TEXT: {response_text}')
    return response_text



In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None): #,stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages
    }
    if system:
        params["system"] = system
    # if stop_sequences:
    #     params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response_content_text(response.content)


In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def grade_by_model(test_case, solution): #grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    # add_assistant_message(messages, "```json")

    eval_text = chat(messages) #, stop_sequences=["```"])

    return json.loads(eval_text) # deserialize the JSON string into a Python dictionary

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    # score = 10
    model_grade = grade_by_model(test_case, output) 
    score = model_grade["score"] 

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results]) 
    print(f"Average score: {average_score}") 
        
    return results

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

The response contains a TextBlock.
RESPONSE TEXT: 2 + 2 = **4**
The response contains a ThinkingBlock.
RESPONSE TEXT: {
  "strengths": [
    "The answer is correct: 2 + 2 = 4.",
    "The response is concise and directly answers the question with no unnecessary content."
  ],
  "weaknesses": [
    "The response provides no explanation or working, which is unnecessary for this trivial question but could matter if the user wanted to understand the reasoning.",
    "The bold formatting on the answer is superfluous for a single-line response."
  ],
  "reasoning": "The solution is mathematically correct and appropriately brief for a trivial arithmetic question. It fully addresses the task. The weaknesses are minor: the lack of explanation is acceptable given the question's simplicity, and the bold formatting is a stylistic choice rather than an error. The score reflects a correct, clear answer with only negligible room for improvement.",
  "score": 8
}

The response contains a TextBlock.
RES